# 03. Events and the meter

The detector on the real test year and on the known events, then the meter's clusters and the
time of use response. The dead end kept here: a detector graded on the known events without a
persistence rule, which would have raised an alert on every noisy hour.

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values = manifest["values"]
v = lambda key: values[key]["value"]
print("as of", manifest["as_of"], "with", len(values), "values")

## The operating point

In [ ]:
point = json.loads((ROOT / "results" / "events" / "operating_point.json").read_text())
print({k: point[k] for k in ("threshold", "persistence_hours", "false_alarm_cost", "missed_event_cost", "cost", "interior")})
print("costs by threshold:", point["costs"])

## The known events

In [ ]:
known = pl.read_parquet(ROOT / "results" / "events" / "known_events.parquet")
print(known.select("event", "node", "onset_utc", "detected", "delay_hours", "peak_z", "false_alarms_in_window"))
print("recall", v("events.known.recall"), "missed:", v("events.known.missed_list"))

## The dead end: no persistence rule

With the persistence rule removed (one hour past the threshold is an alert), the same standardized
residuals in the known event windows raise many more alerts outside the events. The stated rule
costs a few hours of delay and buys the false alarm rate on the page.

In [ ]:
from lookahead_events.detect import find_alerts

windows = pl.read_parquet(ROOT / "results" / "events" / "known_event_windows.parquet")
frame = windows.with_columns(pl.col("flagged").fill_null(False)).rename({"utc_hour": "utc_hour"})
frame = frame.with_columns(pl.lit(None, dtype=pl.Float64).alias("residual_center"), pl.lit(None, dtype=pl.Float64).alias("residual_scale"), pl.lit(None, dtype=pl.Float64).alias("residual"))
with_rule = find_alerts(frame, point["threshold"], point["persistence_hours"])
without = find_alerts(frame, point["threshold"], 1)
print("demand event alerts in the known event windows: with the rule", sum(a.kind == "demand_event" for a in with_rule), "without", sum(a.kind == "demand_event" for a in without))

## The meter

In [ ]:
clusters = pl.read_parquet(ROOT / "results" / "meter" / "clusters.parquet")
print(clusters)
plan = json.loads((ROOT / "results" / "meter" / "plan.json").read_text())
print("plan hash", plan["sha256"][:16], "events", plan["events"], "pairs", plan["pairs"])
print("response", v("meter.tou.response_pct"), "interval", v("meter.tou.response_pct_lower"), v("meter.tou.response_pct_upper"))
print(pl.read_parquet(ROOT / "results" / "meter" / "tou_by_type.parquet"))